# Prática 4 · Prever o caudal daqui a 7 dias

**VERSÃO DO FACILITADOR: o código escrito, com notas.** Não copiar para os computadores dos alunos.

**15 minutos (13:05–13:20) · em pares, sozinhos** · depois da Aula 4

**A missão:** prever o caudal **daqui a 7 dias** (na aula eram 3), escrevendo vocês o alvo, a separação no tempo, o treino e o erro. Depois, ver o que acontece sem o caudal de hoje.

**Os dados:** `caudal_e_chuva_diarios.csv`.  
**Treino e teste:** a mesma da Aula 4: **treino** = 2006 a 2018, **teste** = 2019 a 2026; o alvo é o caudal 7 dias depois.  
**O algoritmo:** regressão linear (regressão: prever um número).

**Como praticar:** agora são vocês que escrevem o código. Em cada passo há uma célula com uma **pista**
(linhas que começam por `#`). Escrevam o código por baixo da pista (podem copiar a pista, tirar o `#` e trocar os
`...`), corram a célula com **Shift + Enter**, e depois corram a célula de verificação: diz «Certo!», ou
«Ainda não» com uma pista. A aula guiada que acabaram de ver tem os mesmos passos: usem-na como modelo.

> **Nota para o facilitador.** Resultados esperados: erro médio no teste a 7 dias 38,8 m³/s (a 3 dias era 26,3); «fica igual a hoje» a 7 dias erra 46,3; só com a chuva, 52,1 m³/s.

## Passo 1 · Abrir as ferramentas e os dados

In [1]:
import os, sys
for pasta in [".", os.path.join("..", "..", "2_pratica")]:          # a pasta onde está o ferramentas.py
    if os.path.exists(os.path.join(pasta, "ferramentas.py")):
        sys.path.insert(0, pasta)
        break
from ferramentas import *
from sklearn.linear_model import LinearRegression
dias = pd.read_csv(encontrar("caudal_e_chuva_diarios.csv"), parse_dates=["data"])
PISTAS = ["caudal_m3s", "chuva_7d_mm", "chuva_14d_mm", "chuva_30d_mm"]
ALVO = "caudal_daqui_a_7_dias"
print("Ferramentas prontas.")

Ferramentas prontas.


## Passo 2 · O alvo: o caudal daqui a 7 dias

Na Aula 4 (Passo 3), o alvo era `shift(-3)`. Aqui são 7 dias.

In [2]:
dias["caudal_daqui_a_7_dias"] = dias["caudal_m3s"].shift(-7)

In [3]:
verificar_passo("4.2")      # a verificação: corram sem mudar nada

Certo! A coluna do alvo tem o caudal de 7 dias depois.


## Passo 3 · Separar no tempo

Tirem as linhas vazias e separem: **treino até 2018**, **teste de 2019 em diante** (Aula 4, Passo 4).

In [4]:
dias = dias.dropna()
treino = dias[dias["data"].dt.year <= 2018]
teste = dias[dias["data"].dt.year >= 2019]

In [5]:
verificar_passo("4.3")      # a verificação: corram sem mudar nada

Certo! 4 719 dias de treino (até 2018) e 2 791 de teste (de 2019 em diante).


## Passo 4 · Treinar a regressão linear

Aula 4, Passo 5.

In [6]:
linear = LinearRegression()
linear.fit(treino[PISTAS], treino[ALVO])

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](4,)","[ 0.52, 0.96,-0.34, 0.38]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](4,)","['caudal_m3s','chuva_7d_mm','chuva_14d_mm','chuva_30d_mm']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-6.535
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(4)


In [7]:
verificar_passo("4.4")      # a verificação: corram sem mudar nada

Certo! A regressão linear foi treinada com o treino.


## Passo 5 · O erro no teste

Calculem o erro médio no teste em `erro_teste`.

In [8]:
erro_teste = (teste[ALVO] - linear.predict(teste[PISTAS])).abs().mean()
print("Erro médio a 7 dias:", numero(erro_teste, 1), "m³/s")

Erro médio a 7 dias: 38,8 m³/s


In [9]:
verificar_passo("4.5")      # a verificação: corram sem mudar nada

Certo! A 7 dias, o modelo erra em média 38,8 m³/s.


## Passo 6 · Sem o caudal de hoje

Treinem outra regressão só com a chuva (`SO_CHUVA`) e calculem o erro no teste em `erro_so_chuva`.

In [10]:
SO_CHUVA = ["chuva_7d_mm", "chuva_14d_mm", "chuva_30d_mm"]
so_chuva = LinearRegression().fit(treino[SO_CHUVA], treino[ALVO])
erro_so_chuva = (teste[ALVO] - so_chuva.predict(teste[SO_CHUVA])).abs().mean()
print("Só com a chuva:", numero(erro_so_chuva, 1), "m³/s")

Só com a chuva: 52,1 m³/s


In [11]:
verificar_passo("4.6")      # a verificação: corram sem mudar nada

Certo! Só com a chuva, o erro sobe para 52,1 m³/s.


## Para pensar

Porque é que o teste usa os anos 2019 a 2026 e não dias escolhidos ao acaso?

a) porque queremos saber se o modelo prevê o futuro, que ele nunca viu  
b) por acaso

In [12]:
resposta = "a"
verificar("P4.a", resposta)

Certo! Um modelo de previsão vai ser usado no futuro. Por isso testa-se com anos que ele nunca viu.


Prever 7 dias à frente, comparado com 3 dias (Aula 4: 26,3 m³/s), é...

a) mais difícil: o erro é maior  
b) mais fácil

In [13]:
resposta = "a"
verificar("P4.b", resposta)

Certo! Quanto mais longe no futuro, mais coisas podem mudar: a previsão a 7 dias erra mais do que a 3.


Qual é a pista mais importante para prever o caudal?

a) o caudal de hoje  
b) a chuva de 30 dias

In [14]:
resposta = "a"
verificar("P4.c", resposta)

Certo! Sem o caudal de hoje, o erro sobe muito: o rio de amanhã parece-se muito com o rio de hoje.


Na Aula 4, a árvore com profundidade 20 tinha erro quase 0 no treino e maior no teste. Como se chama isto?

a) sobreajuste  
b) subajuste  
c) agrupamento

In [15]:
resposta = "a"
verificar("P4.d", resposta)

Certo! Sobreajuste: o modelo decora os exemplos do treino e falha nos dados novos.


## O que aprendemos

- Escreveram sozinhos uma **regressão**: o alvo, a separação no **tempo**, o treino e o **erro médio**.
- Quanto mais longe no futuro, maior o erro; e o caudal de hoje é a pista mais importante.

## Fim da prática: preparar a apresentação (13:20–13:25)

Guardem o trabalho (**Ctrl + S**). Na apresentação, cada par diz em 1 minuto:

1. a **regra** que a árvore aprendeu no vosso ano (Prática 1);
2. o **acerto no teste** e os **falsos alarmes** (Prática 2);
3. o **acerto sem e com a maré** (Prática 3);
4. o **erro a 7 dias** e sem o caudal de hoje (Prática 4);
5. uma coisa que vos surpreendeu.

## Guardar os modelos e as separações (só na versão do facilitador)

Ficam nas pastas `modelos_treinados/` e `separacoes/`, ao lado deste notebook. Para usar um modelo noutro notebook: `modelo = joblib.load("modelos_treinados/...joblib")`.

In [16]:
import joblib
os.makedirs("modelos_treinados", exist_ok=True)
os.makedirs("separacoes", exist_ok=True)
joblib.dump(linear, "modelos_treinados/pratica4_regressao_7_dias.joblib")
treino.to_csv("separacoes/pratica4_treino_2006_a_2018.csv", index=False)
teste.to_csv("separacoes/pratica4_teste_2019_a_2026.csv", index=False)
print("Guardado em modelos_treinados/ e separacoes/.")

Guardado em modelos_treinados/ e separacoes/.


## Se aparecer um erro

Leiam a **última linha** da mensagem vermelha.

| A última linha diz | O que fazer |
| --- | --- |
| `NameError: name '...' is not defined` | Saltaram uma célula. Corram as células por ordem, a partir do Passo 1. |
| `FileNotFoundError: Não encontrei ...` | Abram o notebook a partir da pasta `Workshop_2/2_pratica`. |
| `ModuleNotFoundError` | Falta uma biblioteca neste computador: chamem um facilitador. |
| `NotFittedError` ou «O modelo ainda não foi treinado» | Falta treinar o modelo: corram a célula com `.fit(...)`. |
| `SyntaxError` | Falta uma aspa, um parêntese ou uma vírgula. Comparem com o exemplo. |
| «Falta preencher» | Troquem os `...` pela vossa resposta. |

Se nada resultar: menu **Kernel**, depois **Restart Kernel and Run All Cells**.